## Testing out parenx skeletonization and voronoi approaches

Resources:
* https://github.com/nptscot/networkmerge
* https://github.com/nptscot/networkmerge
* https://github.com/anisotropi4/parenx/tree/main

In [ ]:
import os
import geopandas as gpd
from core import utils
import shutil

In [ ]:
# parquet format is not recognized by parenx - convert to gpkg first
folders = [f"../data/{fua}" for fua in list(utils.fua_city.keys())]

In [ ]:
# make parenx subdirectories (also for temporary gpkg version, which we will later delete)
for folder in folders:
    os.makedirs(folder + "/parenx/", exist_ok=True)
for folder in folders:
    os.makedirs(folder + "/parenx/temp-gpkg/", exist_ok=True)

In [ ]:
# read parquets and save as gpkgs
for fua, case in utils.fua_city.items():
    roads = utils.read_no_degree_2(case)
    roads.to_file(
        f"../data/{fua}/parenx/temp-gpkg/roads_osm.gpkg", layer="roads", engine="pyogrio"
    )

**Now, run the bash script `parenx-run.sh` from command line**

`bash code/parenx-run.sh`

this will add to each `parenx/temp-gpkg` folder 2 files: voronoi.gpkg and skeletonize.gpkg. gitignoring them for now because the outputs are too large; postprocessing below

**reduce output file size by removing duplicated data**,  and copy to corresponding `data/{fua_id]}/parenx/` folders (in parquet format)

In [ ]:
for fua in utils.city_fua.values():

    parenx_folder = f"../data/{fua}/parenx" 

    ske = gpd.read_file(
        filename = parenx_folder + "/temp-gpkg/skeletonize.gpkg",
        driver = "fiona",
        layer = "line"
    )
    ske.to_parquet(parenx_folder + "/skeletonize.parquet")

    vor = gpd.read_file(
        filename = parenx_folder + "/temp-gpkg/voronoi.gpkg",
        driver = "fiona",
        layer = "line"
    )

    vor.to_parquet(parenx_folder + "/voronoi.parquet")

**error logs:**

* 4617 skel ok, vor no
* 4881 skel ok, vor no

```
(simplification) anvy@mac622265  simplification % bash notebooks/parenx-run.sh
./data/1133/parenx/temp-gpkg ./data/1656/parenx/temp-gpkg ./data/4617/parenx/temp-gpkg ./data/4881/parenx/temp-gpkg ./data/809/parenx/temp-gpkg ./data/869/parenx/temp-gpkg ./data/8989/parenx/temp-gpkg
(simplification) anvy@mac622265  simplification % bash notebooks/parenx-run.sh
./data/1133/parenx/temp-gpkg ./data/1656/parenx/temp-gpkg ./data/4617/parenx/temp-gpkg ./data/4881/parenx/temp-gpkg ./data/809/parenx/temp-gpkg ./data/869/parenx/temp-gpkg ./data/8989/parenx/temp-gpkg
Simplification for ./data/1133/parenx/temp-gpkg started
start           0:00:00.000354
read geojson    0:00:00.758024
process         0:00:01.044369
write simple    0:15:40.816585
write primal    0:15:41.808949
stop            0:15:43.863988
start           0:00:00.000178
read geojson    0:00:00.940951
process         0:00:01.270642
dewhisker       0:11:06.199871
write simple    1:41:31.222116
write primal    1:41:33.307194
stop            1:41:33.655133
Simplification for ./data/1656/parenx/temp-gpkg started
start           0:00:00.001229
read geojson    0:00:00.444591
process         0:00:00.604187
write simple    0:22:34.198388
write primal    0:22:34.570195
stop            0:22:35.259949
start           0:00:00.000143
read geojson    0:00:00.366683
process         0:00:00.536325
dewhisker       0:05:02.128189
write simple    0:45:58.165309
write primal    0:45:59.269790
stop            0:45:59.439302
Simplification for ./data/4617/parenx/temp-gpkg started
start           0:00:00.000647
read geojson    0:00:00.620308
process         0:00:00.855644
write simple    0:47:31.827746
write primal    0:47:32.557362
stop            0:47:33.463527
start           0:00:00.000165
read geojson    0:00:00.400152
process         0:00:00.539476
Traceback (most recent call last):
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/ops.py", line 208, in voronoi_diagram
    result = shapely.voronoi_polygons(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/decorators.py", line 77, in wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/constructive.py", line 990, in voronoi_polygons
    return lib.voronoi_polygons(geometry, tolerance, extend_to, only_edges, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
shapely.errors.GEOSException: IllegalArgumentException: point array must contain 0 or >1 elements


The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/anvy/anaconda3/envs/simplification/bin/voronoi.py", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/parenx/voronoi.py", line 247, in main
    nx_voronoi = get_voronoi(nx_boundary, parameter["tolerance"], parameter["scale"])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/parenx/voronoi.py", line 133, in get_voronoi
    r = voronoi_diagram(point, envelope=boundary, tolerance=tolerance, edges=True)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/ops.py", line 216, in voronoi_diagram
    raise ValueError(errstr) from err
ValueError: Could not create Voronoi Diagram with the specified inputs (IllegalArgumentException: point array must contain 0 or >1 elements
). Try running again with default tolerance value.
Simplification for ./data/4881/parenx/temp-gpkg started
start           0:00:00.000389
read geojson    0:00:00.329499
process         0:00:00.454631
write simple    0:07:31.352402
write primal    0:07:31.787947
stop            0:07:32.530303
start           0:00:00.000198
read geojson    0:00:00.329677
process         0:00:00.458156
Traceback (most recent call last):
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/ops.py", line 208, in voronoi_diagram
    result = shapely.voronoi_polygons(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/decorators.py", line 77, in wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/constructive.py", line 990, in voronoi_polygons
    return lib.voronoi_polygons(geometry, tolerance, extend_to, only_edges, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
shapely.errors.GEOSException: IllegalArgumentException: point array must contain 0 or >1 elements


The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/anvy/anaconda3/envs/simplification/bin/voronoi.py", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/parenx/voronoi.py", line 247, in main
    nx_voronoi = get_voronoi(nx_boundary, parameter["tolerance"], parameter["scale"])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/parenx/voronoi.py", line 133, in get_voronoi
    r = voronoi_diagram(point, envelope=boundary, tolerance=tolerance, edges=True)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/anvy/anaconda3/envs/simplification/lib/python3.11/site-packages/shapely/ops.py", line 216, in voronoi_diagram
    raise ValueError(errstr) from err
ValueError: Could not create Voronoi Diagram with the specified inputs (IllegalArgumentException: point array must contain 0 or >1 elements
). Try running again with default tolerance value.
Simplification for ./data/809/parenx/temp-gpkg started
start           0:00:00.000382
read geojson    0:00:00.667745
process         0:00:00.909928
write simple    0:07:24.202551
write primal    0:07:24.826585
stop            0:07:26.413387
start           0:00:00.000180
read geojson    0:00:00.662064
process         0:00:00.907475
dewhisker       0:06:25.726934
write simple    1:04:40.892776
write primal    1:04:42.194655
stop            1:04:42.393156
Simplification for ./data/869/parenx/temp-gpkg started
start           0:00:00.000434
read geojson    0:00:00.297471
process         0:00:00.406435
write simple    0:04:04.664959
write primal    0:04:04.899290
stop            0:04:05.370182
start           0:00:00.000175
read geojson    0:00:00.305808
process         0:00:00.418956
dewhisker       0:02:37.557307
write simple    0:20:43.222941
write primal    0:20:43.665956
stop            0:20:43.740963
Simplification for ./data/8989/parenx/temp-gpkg started
start           0:00:00.000388
read geojson    0:00:00.597390
process         0:00:00.823545
write simple    3:54:03.572635
write primal    3:54:04.738093
stop            3:54:06.305120
start           0:00:00.000214
read geojson    0:00:00.615457
process         0:00:00.842010
dewhisker       0:25:03.548439
write simple    3:05:52.589831
write primal    3:05:54.136019
stop            3:05:54.372537
Done.
```

In [ ]:
from functools import partial
from shapely.ops import voronoi_diagram
from shapely.geometry import LineString, MultiPoint, Point
from shapely import box, get_coordinates, unary_union, set_precision
import numpy as np
import pandas as pd
from pyogrio import read_dataframe

In [ ]:
fua = 4617
filepath = f"../data/{fua}/parenx/temp-gpkg/roads_osm.gpkg"
tolerance = 1.0 # meters
scale = 5.0 # voronoi scale
radius = 8.0 # meters
mycrs = gpd.read_file(filepath).crs

In [ ]:
set_precision_pointone = partial(set_precision, grid_size=0.1)

def get_base_geojson(filepath, mycrs):
    """get_base_nx: return GeoDataFrame at 0.1m precision from GeoJSON

    args:
      filepath: GeoJSON path

    returns:
      GeoDataFrame at 0.1m precision

    """
    r = read_dataframe(filepath).to_crs(mycrs)
    r["geometry"] = r["geometry"].map(set_precision_pointone)
    return r

def get_geometry_buffer(this_gf, mycrs, radius=8.0):
    """get_geometry_buffer: return radius buffered GeoDataFrame

    args:
      this_gf: GeoDataFrame to
      radius: (default value = 8.0)

    returns:
      buffered GeoSeries geometry

    """
    try:
        r = gpd.GeoSeries(unary_union(this_gf).geoms, crs=mycrs)
    except AttributeError:
        r = gpd.GeoSeries(this_gf, crs=mycrs)
    r = gpd.GeoSeries(r, crs=mycrs).buffer(radius, join_style="mitre")
    union = unary_union(r)
    try:
        r = gpd.GeoSeries(union.geoms, crs=mycrs)
    except AttributeError:
        r = gpd.GeoSeries(union, crs=mycrs)
    return r

def get_linestring(line, mycrs):
    """get_linestring: return LineString GeoSeries from line coordinates

    args:
      line:

    returns:
       LineString GeoSeries
    """
    r = get_coordinates(line)
    r = np.stack([gpd.points_from_xy(*r[:-1].T), gpd.points_from_xy(*r[1:].T)])
    return gpd.GeoSeries(pd.DataFrame(r.T).apply(LineString, axis=1), crs=mycrs).values

def get_segment(line, mycrs, distance=50.0):
    """get_segment: segment LineString GeoSeries into distance length segments

    args:
      line: GeoSeries LineString
      length: segmentation distance (default value = 50.0)

    returns:
      GeoSeries of LineStrings of up to length distance

    """
    return get_linestring(line.segmentize(distance), mycrs)

def get_segment_nx(line, scale, mycrs):
    """get_segment_nx: segment line into sections, no more than scale long

    args:
      line:  line to segment
      scale: length to segment line

    returns:
      segmented LineStrings

    """
    set_segment = partial(get_segment, mycrs=mycrs, distance=scale)
    r = line.map(set_segment).explode().rename("geometry")
    return gpd.GeoDataFrame(r, crs=mycrs)

def get_geometry_line(this_buffer, mycrs):
    """get_geometry_line: returns LineString boundary from geometry

    args:
      this_buffer: geometry to find LineString

    returns:
       simplified LineString boundary
    """
    r = this_buffer.boundary.explode(index_parts=False).reset_index(drop=True)
    return gpd.GeoSeries(r.simplify(tolerance=0.5), crs=mycrs)

In [ ]:
base_nx_geometry =  get_base_geojson(filepath, mycrs)["geometry"]

In [ ]:
nx_geometry = get_geometry_buffer(base_nx_geometry, mycrs=mycrs, radius=radius)

In [ ]:
nx_boundary = get_geometry_line(nx_geometry, mycrs)

In [ ]:
segment = get_segment_nx(nx_boundary, mycrs=mycrs, scale=scale).reset_index(drop=True)

In [ ]:
point = segment.loc[:, "geometry"].map(get_coordinates).explode()

In [ ]:
point = MultiPoint(point[::2].map(Point).values)

In [ ]:
boundary = box(*point.bounds)

In [ ]:
r = voronoi_diagram(point, envelope=boundary, tolerance=tolerance, edges=True)

In [ ]:
r = gpd.GeoSeries(map(set_precision_pointone, r.geoms), crs=mycrs)

In [ ]:
r = r.explode(index_parts=False).clip(boundary)

In [ ]:
ix = ~r.is_empty & (r.type == "LineString")
r = r[ix].reset_index(drop=True)

In [ ]:
r.head()

***
***

**Remove files in temp-gpkg folder**

In [ ]:
for fua in utils.city_fua.values():
    obsolete_folder = f"../data/{fua}/parenx/temp-gpkg/"
    shutil.rmtree(obsolete_folder)

### Initial observations & thoughts:
* computation time: skeletonization around 10min for all 5 usecases; voronoi between 1h and 14h (salt lake city, maybe because it has the largest area, or maybe because my laptop went to sleep...)
*  it works well for some places (esp intersections, even the more complicated ones)
* major issue 1: sometimes network topology is not kept (linestrings that don't connect are merged)
* major issue 2: it creates wobbly lines